# 01 — Exploratory Data Analysis
**Project:** E-Commerce Fraud Decisioning & Analytics Engine  
**Dataset:** IEEE-CIS Fraud Detection (Kaggle)  
**Author:** Avraham Koslowsky  

---

## Goals
1. **Class imbalance** — understand the fraud rate and its implications for model evaluation.
2. **Feature missingness map** — identify columns with high null rates; some missingness is itself a fraud signal.
3. **Temporal patterns** — how does fraud rate evolve over `TransactionDT`? Are there intra-day or weekly cycles?
4. **Transaction amount distribution** — fraud vs. legitimate amount profiles.
5. **Identity join coverage** — what fraction of transactions have identity enrichment?
6. **V-feature structure** — PCA snapshot of the 339 anonymised Vesta features.

> **Notebook convention:** All heavy computation reads from `data/interim/` (post-merge parquet).  
> If that file doesn't exist yet, the first cell will generate it.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Make project root importable
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.loader import load_and_merge

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FRAUD_COLOR   = "#e63946"   # red   — fraudulent
LEGIT_COLOR   = "#457b9d"   # blue  — legitimate
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name: str, tight: bool = True) -> None:
    """Save current figure to reports/figures/ and display inline."""
    path = FIGURES_DIR / name
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved → {path}")

print("Imports OK.")

## 0 — Load Data

In [ ]:
INTERIM_PARQUET = PROJECT_ROOT / "data" / "interim" / "merged.parquet"

if INTERIM_PARQUET.exists():
    print("Loading from cached parquet …")
    df = pd.read_parquet(INTERIM_PARQUET)
else:
    print("Parquet not found — running full ingestion (may take ~60 s) …")
    df = load_and_merge(optimize_memory=True)
    INTERIM_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(INTERIM_PARQUET, index=False)
    print(f"Cached to {INTERIM_PARQUET}")

print(f"\nShape   : {df.shape}")
print(f"Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head(3)

## 1 — Class Imbalance

The dataset is heavily imbalanced (~3.5% fraud).  
Key takeaway: **we will optimise PR-AUC, not accuracy or ROC-AUC**, and will use `scale_pos_weight` in gradient boosting.

In [ ]:
counts     = df["isFraud"].value_counts()
fraud_rate = df["isFraud"].mean()
imbalance_ratio = counts[0] / counts[1]

print(f"Legitimate  : {counts[0]:>7,}  ({1-fraud_rate:.2%})")
print(f"Fraud       : {counts[1]:>7,}  ({fraud_rate:.2%})")
print(f"Imbalance   : {imbalance_ratio:.1f}x (legit : fraud)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart — raw counts
axes[0].bar(
    ["Legitimate", "Fraud"],
    [counts[0], counts[1]],
    color=[LEGIT_COLOR, FRAUD_COLOR],
    edgecolor="white", linewidth=1.2,
)
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 2000, f"{v:,}", ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Transaction Count by Class", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

# Pie chart — proportions
axes[1].pie(
    [counts[0], counts[1]],
    labels=[f"Legitimate\n{1-fraud_rate:.2%}", f"Fraud\n{fraud_rate:.2%}"],
    colors=[LEGIT_COLOR, FRAUD_COLOR],
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor="white"),
    textprops=dict(fontsize=11),
)
axes[1].set_title("Class Proportion", fontweight="bold")

fig.suptitle("Class Imbalance — IEEE-CIS Fraud Dataset", fontsize=13, fontweight="bold", y=1.02)
savefig("01_class_imbalance.png")
plt.show()

## 2 — Feature Missingness Map

**Key insight:** Missingness is not random here. Many `id_*` columns are missing for ~75% of rows
(transactions without identity enrichment). High missingness in a feature ≠ useless feature —  
the *pattern* of missingness is itself a fraud signal.

In [ ]:
# ── Overall null rate per column ────────────────────────────────────────────
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

print(f"Columns with ANY nulls : {(null_pct > 0).sum()} / {len(null_pct)}")
print(f"Columns >50% null      : {(null_pct > 50).sum()}")
print(f"Columns >90% null      : {(null_pct > 90).sum()}")
print(f"\nTop 15 highest-null columns:")
null_pct.head(15).to_frame(name="null_%").round(1)

In [ ]:
# ── Heatmap: null rate bucketed by column group ─────────────────────────────
# Group columns into logical families for readability
COLUMN_FAMILIES = {
    "Core": ["TransactionID", "TransactionDT", "TransactionAmt",
              "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
              "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain"],
    "M features": [c for c in df.columns if c.startswith("M")],
    "C features": [c for c in df.columns if c.startswith("C")],
    "D features": [c for c in df.columns if c.startswith("D")],
    "V features (1-100)": [f"V{i}" for i in range(1, 101) if f"V{i}" in df.columns],
    "V features (101-200)": [f"V{i}" for i in range(101, 201) if f"V{i}" in df.columns],
    "V features (201-339)": [f"V{i}" for i in range(201, 340) if f"V{i}" in df.columns],
    "Identity": [c for c in df.columns if c.startswith("id_") or c.startswith("DeviceType") or c.startswith("DeviceInfo")],
}

# Build a summary DataFrame: one row per family
family_null = {}
for family, cols in COLUMN_FAMILIES.items():
    present_cols = [c for c in cols if c in df.columns]
    if present_cols:
        family_null[family] = null_pct[present_cols].values

fig, axes = plt.subplots(len(family_null), 1,
                          figsize=(16, len(family_null) * 1.6),
                          gridspec_kw={"hspace": 0.05})

for ax, (family, vals) in zip(axes, family_null.items()):
    hm_data = vals.reshape(1, -1)
    sns.heatmap(
        hm_data,
        ax=ax, vmin=0, vmax=100,
        cmap="YlOrRd",
        cbar=(family == list(family_null.keys())[-1]),
        xticklabels=False, yticklabels=[family],
        linewidths=0,
    )
    # Annotate mean null rate
    ax.text(1.01, 0.5, f"mean={vals.mean():.0f}%",
            transform=ax.transAxes, va="center", fontsize=9, color="#555")

fig.suptitle("Feature Missingness Map (% null per column, grouped by family)",
             fontsize=13, fontweight="bold", y=1.01)
savefig("02_missingness_map.png")
plt.show()

In [ ]:
# ── Missingness vs. fraud rate: is missing-ness itself predictive? ──────────
# For each column with >5% nulls, compare fraud rate in null vs. non-null rows

high_null_cols = null_pct[null_pct > 5].index.tolist()
high_null_cols = [c for c in high_null_cols if c != "isFraud"]

records = []
for col in high_null_cols[:40]:   # cap at 40 for display
    is_null   = df[col].isnull()
    fraud_if_null    = df.loc[is_null,  "isFraud"].mean()
    fraud_if_present = df.loc[~is_null, "isFraud"].mean()
    records.append({
        "column": col,
        "null_pct": null_pct[col],
        "fraud_rate_if_null": fraud_if_null,
        "fraud_rate_if_present": fraud_if_present,
        "delta": fraud_if_null - fraud_if_present,
    })

miss_df = pd.DataFrame(records).sort_values("delta", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [FRAUD_COLOR if d > 0 else LEGIT_COLOR for d in miss_df["delta"]]
ax.barh(miss_df["column"], miss_df["delta"] * 100, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ Fraud Rate (null rows − present rows)  [pp]")
ax.set_title("Is Missingness Itself a Fraud Signal?", fontweight="bold")
ax.invert_yaxis()
savefig("03_missingness_vs_fraud.png")
plt.show()

print("Top 5 columns where missing → higher fraud rate:")
miss_df.head(5)[["column", "null_pct", "fraud_rate_if_null", "fraud_rate_if_present", "delta"]]

## 3 — Temporal Patterns in Fraud Rate

`TransactionDT` is a second-level delta from an unknown epoch.  
We convert it to relative hours and days to expose cyclical fraud patterns.

In [ ]:
# ── Derive temporal features from TransactionDT ─────────────────────────────
DT = df["TransactionDT"]

df["_hour_of_day"]  = (DT // 3600) % 24          # 0–23
df["_day_of_week"]  = (DT // 86400) % 7           # 0–6
df["_week"]         = DT // (7 * 86400)            # week index

print(f"TransactionDT range: {DT.min():,} → {DT.max():,}")
print(f"Approx duration    : {(DT.max() - DT.min()) / 86400:.0f} days")

In [ ]:
# ── Rolling fraud rate over time ─────────────────────────────────────────────
df_sorted = df.sort_values("TransactionDT").copy()

# Bin into 500 equal-width time buckets and compute fraud rate per bucket
df_sorted["_time_bin"] = pd.cut(df_sorted["TransactionDT"], bins=500, labels=False)
temporal = (
    df_sorted.groupby("_time_bin", observed=True)
    .agg(fraud_rate=("isFraud", "mean"), count=("isFraud", "count"),
         dt_mid=("TransactionDT", "median"))
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05})

# Fraud rate (smoothed)
axes[0].plot(temporal["dt_mid"], temporal["fraud_rate"] * 100,
             color=FRAUD_COLOR, linewidth=0.9, alpha=0.7, label="Fraud rate (raw)")
rolling_mean = temporal["fraud_rate"].rolling(20, center=True).mean()
axes[0].plot(temporal["dt_mid"], rolling_mean * 100,
             color=FRAUD_COLOR, linewidth=2.5, label="Fraud rate (20-bin MA)")
axes[0].axhline(df["isFraud"].mean() * 100, color="gray",
                linestyle="--", linewidth=1, label=f"Overall mean ({df['isFraud'].mean():.2%})")
axes[0].set_ylabel("Fraud Rate (%)")
axes[0].set_title("Fraud Rate Over Time (TransactionDT)", fontweight="bold")
axes[0].legend()

# Volume
axes[1].bar(temporal["dt_mid"], temporal["count"], width=temporal["dt_mid"].diff().median(),
            color=LEGIT_COLOR, alpha=0.6, label="Transaction volume")
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("TransactionDT (seconds)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
axes[1].legend()

savefig("04_temporal_fraud_rate.png")
plt.show()

In [ ]:
# ── Hour-of-day fraud rate ────────────────────────────────────────────────────
hourly = df.groupby("_hour_of_day")["isFraud"].agg(["mean", "count"]).reset_index()
hourly.columns = ["hour", "fraud_rate", "count"]

fig, ax1 = plt.subplots(figsize=(13, 4))
ax2 = ax1.twinx()

ax1.bar(hourly["hour"], hourly["count"] / 1000, color=LEGIT_COLOR, alpha=0.5, label="Volume (k)")
ax2.plot(hourly["hour"], hourly["fraud_rate"] * 100, color=FRAUD_COLOR,
         marker="o", linewidth=2, markersize=5, label="Fraud rate %")

ax1.set_xlabel("Hour of Day (relative)")
ax1.set_ylabel("Volume (thousands)", color=LEGIT_COLOR)
ax2.set_ylabel("Fraud Rate (%)", color=FRAUD_COLOR)
ax1.set_xticks(range(24))
ax1.set_title("Fraud Rate by Hour of Day", fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

savefig("05_hourly_fraud_rate.png")
plt.show()

peak_hour = hourly.loc[hourly["fraud_rate"].idxmax(), "hour"]
print(f"Peak fraud hour: {peak_hour}:00  (fraud rate {hourly.loc[hourly['fraud_rate'].idxmax(), 'fraud_rate']:.2%})")

In [ ]:
# ── Day-of-week fraud rate ────────────────────────────────────────────────────
daily = df.groupby("_day_of_week")["isFraud"].agg(["mean", "count"]).reset_index()
daily.columns = ["dow", "fraud_rate", "count"]
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily["dow_label"] = daily["dow"].map(lambda x: dow_labels[x])

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.bar(daily["dow_label"], daily["count"] / 1000, color=LEGIT_COLOR, alpha=0.5)
ax2.plot(daily["dow_label"], daily["fraud_rate"] * 100,
         color=FRAUD_COLOR, marker="o", linewidth=2.2, markersize=7)

ax1.set_ylabel("Volume (k)", color=LEGIT_COLOR)
ax2.set_ylabel("Fraud Rate (%)", color=FRAUD_COLOR)
ax1.set_title("Fraud Rate by Day of Week", fontweight="bold")

savefig("06_daily_fraud_rate.png")
plt.show()

## 4 — Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

legit_amt = df.loc[df["isFraud"] == 0, "TransactionAmt"]
fraud_amt = df.loc[df["isFraud"] == 1, "TransactionAmt"]

# Log-scale histogram
bins = np.logspace(np.log10(0.5), np.log10(df["TransactionAmt"].max() + 1), 80)
axes[0].hist(legit_amt, bins=bins, color=LEGIT_COLOR, alpha=0.6, label="Legitimate", density=True)
axes[0].hist(fraud_amt, bins=bins, color=FRAUD_COLOR, alpha=0.6, label="Fraud",      density=True)
axes[0].set_xscale("log")
axes[0].set_xlabel("TransactionAmt (log scale)")
axes[0].set_ylabel("Density")
axes[0].set_title("Amount Distribution (log scale)", fontweight="bold")
axes[0].legend()

# Box plot
axes[1].boxplot(
    [legit_amt.clip(upper=2000), fraud_amt.clip(upper=2000)],
    labels=["Legitimate", "Fraud"],
    patch_artist=True,
    boxprops=dict(facecolor="none"),
    medianprops=dict(linewidth=2.5),
    flierprops=dict(marker=".", markersize=1, alpha=0.3),
)
axes[1].set_ylabel("TransactionAmt (clipped at $2,000)")
axes[1].set_title("Amount Box Plot", fontweight="bold")

fig.suptitle("Transaction Amount: Fraud vs. Legitimate", fontweight="bold", fontsize=13, y=1.02)
savefig("07_transaction_amount.png")
plt.show()

print(f"Fraud   median: ${fraud_amt.median():.2f}  |  mean: ${fraud_amt.mean():.2f}")
print(f"Legit   median: ${legit_amt.median():.2f}  |  mean: ${legit_amt.mean():.2f}")

## 5 — Identity Join Coverage

In [ ]:
# Use id_01 as a proxy for identity record presence
id_proxy_col = next((c for c in df.columns if c.startswith("id_")), None)

if id_proxy_col:
    df["_has_identity"] = df[id_proxy_col].notna().astype(int)

    identity_coverage = df["_has_identity"].mean()
    fraud_with_id    = df.loc[df["_has_identity"] == 1, "isFraud"].mean()
    fraud_without_id = df.loc[df["_has_identity"] == 0, "isFraud"].mean()

    print(f"Transactions WITH identity    : {identity_coverage:.1%}")
    print(f"Fraud rate WITH identity      : {fraud_with_id:.2%}")
    print(f"Fraud rate WITHOUT identity   : {fraud_without_id:.2%}")
    print(f"Delta                         : {(fraud_with_id - fraud_without_id)*100:+.2f} pp")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["With Identity", "Without Identity"],
           [fraud_with_id * 100, fraud_without_id * 100],
           color=[FRAUD_COLOR, LEGIT_COLOR])
    ax.set_ylabel("Fraud Rate (%)")
    ax.set_title("Fraud Rate: Identity Record Present vs. Absent", fontweight="bold")
    for i, v in enumerate([fraud_with_id, fraud_without_id]):
        ax.text(i, v * 100 + 0.05, f"{v:.2%}", ha="center", fontweight="bold")
    savefig("08_identity_coverage.png")
    plt.show()
else:
    print("No identity columns found — skipping this section.")

## 6 — V-Feature Structure (PCA)

There are 339 anonymised Vesta features (V1–V339) which are the richest signal source.  
We use PCA to get a sense of their structure and check if fraud clusters in the principal component space.

In [ ]:
v_cols = [c for c in df.columns if c.startswith("V") and c[1:].isdigit()]
print(f"V-feature count: {len(v_cols)}")

# Use a sample for PCA (full dataset may be slow)
N_SAMPLE = 50_000
sample   = df.sample(n=min(N_SAMPLE, len(df)), random_state=42)

# Impute with median, then scale
X_v = sample[v_cols].fillna(sample[v_cols].median())
X_scaled = StandardScaler().fit_transform(X_v)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Explained variance — PC1: {explained[0]:.2%}, PC2: {explained[1]:.2%}")

fig, ax = plt.subplots(figsize=(9, 6))
mask_legit = sample["isFraud"].values == 0
mask_fraud = sample["isFraud"].values == 1

ax.scatter(X_pca[mask_legit, 0], X_pca[mask_legit, 1],
           s=4, alpha=0.3, color=LEGIT_COLOR, label="Legitimate", rasterized=True)
ax.scatter(X_pca[mask_fraud, 0], X_pca[mask_fraud, 1],
           s=8, alpha=0.6, color=FRAUD_COLOR, label="Fraud", rasterized=True)

ax.set_xlabel(f"PC1 ({explained[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} var)")
ax.set_title("PCA of V-Features (n=50k sample)", fontweight="bold")
ax.legend(markerscale=3)
savefig("09_pca_v_features.png")
plt.show()

## 7 — EDA Summary

Key findings that drive every downstream modelling decision:

In [ ]:
summary = {
    "Total transactions"            : f"{len(df):,}",
    "Fraud rate"                     : f"{df['isFraud'].mean():.4%}",
    "Imbalance ratio (legit:fraud)": f"{(1-df['isFraud'].mean())/df['isFraud'].mean():.1f}x",
    "Total features"                : f"{df.shape[1]}",
    "Columns with any null"         : f"{(df.isnull().any()).sum()}",
    "Columns >50% null"             : f"{(df.isnull().mean() > 0.5).sum()}",
    "V-features"                    : f"{len(v_cols)}",
    "Approx dataset duration"       : f"{(DT.max()-DT.min())/86400:.0f} days",
}
if id_proxy_col:
    summary["Identity join rate"]   = f"{identity_coverage:.1%}"

print("=" * 55)
print("EDA SUMMARY")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:<40} {v}")
print("=" * 55)

print("""
Modelling implications
----------------------
1. METRIC: Use PR-AUC as the primary metric (not ROC-AUC) — the severe
   class imbalance means ROC-AUC can look good while precision collapses.

2. SPLIT: Time-aware split is mandatory — temporal concept drift is visible
   in the rolling fraud rate chart above.

3. MISSINGNESS: Missing identity columns are informative (Δ fraud rate ≠ 0).
   Engineer `has_identity` and per-family null-count features.

4. V-FEATURES: PCA shows partial separation in the first two components,
   validating them as strong predictors.

5. AMOUNT: Fraud skews to higher amounts — log-transformed amount is a
   strong baseline feature.

6. TEMPORAL: Intra-day hour cyclicality → include hour-of-day as a feature.
""")

In [ ]:
# Clean up temporary columns before saving
tmp_cols = [c for c in df.columns if c.startswith("_")]
df.drop(columns=tmp_cols, inplace=True)

print(f"EDA complete. {len(list(FIGURES_DIR.glob('*.png')))} figures saved to {FIGURES_DIR}")